# 2: Pre-Industrial Compset: F1850C_LTso 

## Understanding Pre-Industrial Simulations

In the previous exercise, you created a control simulation using `FHISTC_LTso`, which uses time-varying **historical** sea surface temperatures, sea ice, greenhouse gases, aerosols, and other forcings.

In this exercise, you will run `F1850C_LTso`, a **pre-industrial** configuration. Unlike the historical compset, F1850C_LTso uses repeating climatological conditions representative of approximately the year 1850.

This type of simulation can be used as a baseline pre-industrial climate state for climate change studies because it represents Earth before major anthropogenic influences.

By comparing this simulation to the historical control, you will learn how CESM handles different forcing configurations and how to verify that your simulation is using the intended datasets.


<div class="alert alert-info">
<strong>Exercise: Run a Pre-Industrial Simulation</strong><br><br>
 
Create, configure, build, and run a case called `f1850` using:

- Compset: `F1850C_LTso`
- Resolution: `ne16pg3_ne16pg3_mg17`

Use the same CAM history output settings as in the previous exercise and run for 5 days.
Improve the throughput by changing the task layout as in the control case. 

As you work through the exercise, consider the following questions:

- How can you verify that this simulation is configured differently from your historical control case?
- How can you determine which SST and forcing datasets are being used?
- What model year is being simulated?
- How does the model start date differ from the historical control case?

</div>



<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>
<br>

**How do I output 3 hourly instantaneous variables?**

- Use namelist variables: ``nhtfrq``, ``mfilt``, ``fincl``. 
- For more information, look at the chapter: <br>
**NAMELIST MODIFICATIONS** -> **Customize CAM output**

**How can I verify which forcing datasets,SSTs, and start date are being used?**

Look at the generated namelists:

`$CASEDIR/CaseDocs/atm_in`

You can also compare the SSTs between the historical and pre-industrial cases:

`./xmlquery SSTICE_DATA_FILENAME`

and the start date:

`./xmlquery RUN_STARTDATE`

**I am getting an error: ``This compset and grid combination is untested in CESM``**
You might be getting this error if the `ne16pg3_ne16pg3_mg17` is not a supported resolution in teh version you are using. 

Overide this error by adding ``--run-unsupported`` to the create_newcase command

</details>
</div>



<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
 
 
**# Set environment variables** 
    
Set environment variables with the commands:

**tcsh user**
```
set CASENAME=f1850
set CASEDIR=/glade/u/home/$USER/cases/$CASENAME
set RUNDIR=/glade/derecho/scratch/$USER/$CASENAME/run
set COMPSET=F1850C_LTso
set RESOLUTION=ne16pg3_ne16pg3_mg17
```
**bash user**
```
export CASENAME=f1850
export CASEDIR=/glade/u/home/$USER/cases/$CASENAME
export RUNDIR=/glade/derecho/scratch/$USER/$CASENAME/run
export COMPSET=F1850C_LTso
export RESOLUTION=ne16pg3_ne16pg3_mg17
```    
    
**# Create a new case**

Create a new case with the command ``create_newcase``:
```
cd /glade/u/home/$USER/code/my_cesm_code/cime/scripts/
./create_newcase --case $CASEDIR --res $RESOLUTION --compset $COMPSET --run-unsupported
```

**# Change the job queue and account number**

If needed, change ``job queue`` and ``account number``.<br>
For instance, to run in the queue ``tutorial`` and the project number ``UESM0016``. You should use the project number given for this tutorial.
```  
cd $CASEDIR
./xmlchange JOB_QUEUE=tutorial,PROJECT=UESM0016 --force
```
This step can be redone at anytime in the process. 

**#  Increase model throughput**

To improve throughput, change the task layout before running `case.setup`:

```bash
./xmlchange NTASKS=-6
```


**# Setup**

Invoke ``case.setup`` with the command:
```    
cd $CASEDIR
./case.setup    
```  

**# Customize namelists**

Edit the file ``user_nl_cam`` and add the lines:
```
nhtfrq(2) = -3
mfilt(2) = 240
fincl2 = 'TS:I','PS:I', 'U850:I','T850:I','PRECT:I','LHFLX:I','SHFLX:I','FLNT:I','FLNS:I'
interpolate_output(2) = .true.
interpolate_nlat(2)   =  91
interpolate_nlon(2)   = 180
```
You can do this with a text editor. Alternatively, you can use the echo command:
```
echo "nhtfrq(2) = -3">> user_nl_cam    
echo "mfilt(2) = 240">> user_nl_cam
echo "fincl2 = 'TS:I','PS:I', 'U850:I','T850:I','PRECT:I','LHFLX:I','SHFLX:I','FLNT:I','FLNS:I'">> user_nl_cam
echo "interpolate_output(2) = .true.">> user_nl_cam    
echo "interpolate_nlat(2)   = 91">> user_nl_cam    
echo "interpolate_nlon(2)   = 180">> user_nl_cam    
echo "">> user_nl_cam  
```

You build the namelists with the command:
```
./preview_namelists
```
This step is optional as the script ``preview_namelists`` is automatically called by ``case.build`` and ``case.submit``. But it is usefule to check that your changes made their way into:
```
$CASEDIR/CaseDocs/atm_in
```


**# Set run length**

If needed, change the ``run length``. If you want to run 5 days, you don't have to do this, as 5 days is the default. 
```   
./xmlchange STOP_N=5,STOP_OPTION=ndays
```


**# Build and submit**:
```
qcmd -- ./case.build
./case.submit
```
------------

**# Verify the Configuration**

Before or after the run, compare key settings with your historical control case.

**Check the Start Date**
```
./xmlquery RUN_STARTDATE
```
Unlike the historical case, which starts in a modern historical year, the F1850C_LTso simulation should start at 0001 which is typically when using a pre-industrial repeating a single year. 

**Check the SST Dataset**

You should find that:
- fhist.control uses historical time-varying SST datasets.
- f1850 uses pre-industrial climatological SST datasets representative of approximately 1850.
  
`./xmlquery SSTICE_DATA_FILENAME`

For the  should see: 	
```
SSTICE_DATA_FILENAME: /glade/campaign/cesm/cesmdata/inputdata/atm/cam/sst/sst_HadOIBl_bc_1x1_clim_pi_c101029.nc
```

**Compare Additional Forcing Datasets**

Inspect the atmospheric namelist:
```
 $CASEDIR/CaseDocs/atm_in
```
Compare the forcing datasets used by:
fhist.control
f1850

Look for differences in:
- Greenhouse gas concentrations
- Aerosol forcing
- ...

These differences are the primary reason the two simulations represent different climate states.


**# Check your solution**

When the run is completed, look at the history files into the archive directory.  
    
(1) Check that your archive directory on derecho (The path will be different on other machines): 
```
cd /glade/derecho/scratch/$USER/archive/$CASENAME/atm/hist
ls 
```

As your run is only 5-day, there should be no monthly file (``h0``)

Notice that the history file names begin in year 0001. This is different from fhist.control, which starts in a modern historical period.

(2) Look at the contents of the ``h1`` files using ``ncdump``.
 
- The file should contain the instantaneous output in the file ``h1`` for the variables:
```
        float FLNS(time, lat, lon) ;
                FLNS:Sampling_Sequence = "rad_lwsw" ;
                FLNS:units = "W/m2" ;
                FLNS:long_name = "Net longwave flux at surface" ;
        float FLNT(time, lat, lon) ;
                FLNT:Sampling_Sequence = "rad_lwsw" ;
                FLNT:units = "W/m2" ;
                FLNT:long_name = "Net longwave flux at top of model" ;
        float LHFLX(time, lat, lon) ;
                LHFLX:units = "W/m2" ;
                LHFLX:long_name = "Surface latent heat flux" ;
        float PRECT(time, lat, lon) ;
                PRECT:units = "m/s" ;
                PRECT:long_name = "Total (convective and large-scale) precipitation rate (liq + ice)" ;
        float PS(time, lat, lon) ;
                PS:units = "Pa" ;
                PS:long_name = "Surface pressure" ;
        float SHFLX(time, lat, lon) ;
                SHFLX:units = "W/m2" ;
                SHFLX:long_name = "Surface sensible heat flux" ;
        float T850(time, lat, lon) ;
                T850:units = "K" ;
                T850:long_name = "Temperature at 850 mbar pressure surface" ;
        float TS(time, lat, lon) ;
                TS:units = "K" ;
                TS:long_name = "Surface temperature (radiative)" ;
        float U850(time, lat, lon) ;
                U850:units = "m/s" ;
                U850:long_name = "Zonal wind at 850 mbar pressure surface" ;
```
Note that these variables have no ``cell_methods`` attribute because the output is instantaneous. 


**# What Changed Relative to the Historical Control?**

Compare the following between `fhist.control` and `f1850`:

- RUN_STARTDATE
- SST forcing datasets
- Greenhouse gas concentrations
- Aerosol forcing
- Atmospheric chemistry forcing files
- Sea-ice forcing datasets

Understanding these differences is one of the key goals of this exercise.

The atmospheric model and output settings are nearly identical between the two cases. The primary difference is the forcing environment: one represents the historical climate system, while the other represents pre-industrial conditions.


| Setting          | fhist.control             | f1850.control                      | Why it matters                      |
| ---------------- | ------------------------- | ---------------------------------- | ----------------------------------- |
| Compset          | FHISTC_LTso               | F1850C_LTso                        | Controls the forcing protocol       |
| Resolution       | ne16pg3_ne16pg3_mg17      | ne16pg3_ne16pg3_mg17               | Same model grid                     |
| Forcing Type     | Historical, time-varying  | Pre-industrial climatology         | Main scientific difference          |
| SST Dataset      | Historical SSTs           | Pre-industrial climatological SSTs | Changes atmospheric forcing         |
| Sea Ice          | Historical, time-varying  | Pre-industrial climatology         | Changes surface boundary conditions |
| Greenhouse Gases | Historical concentrations | 1850 concentrations                | Changes radiative forcing           |
| Aerosols         | Historical forcing        | 1850 forcing                       | Changes clouds and radiation        |
| Start Date       | Modern historical period  | Year 0001                          | Different model calendar            |


    
</details>
</div>

